# Notebook 03: Interview Conversation and Follow-up Generation — OFPHIL MSW Interview Assistant

**Project:** `ofphil-classification-v2`  
**Phase:** 05 Model Selection and Training — Track B: LLM Interview Assistant  
**Notebook:** `03_interview_conversation_and_followup_generation.ipynb`  
**Role:** Multi-Turn Dialogue Management, Context Tracking & MSW-in-the-Loop Workflow

---

### Objectives
1. Implement the conversational dialogue state tracker for multi-turn Stage 2 MSW patient interviews.
2. Develop deterministic context window management to prevent prompt bloat and repetitive questions.
3. Simulate realistic synthetic Taglish patient dialogues across multiple domains:
   - Patient expressing irregular informal earnings (e.g., tricycle driving / seasonal vending).
   - Patient reporting prior hospital debt or guarantee letters.
   - Patient struggling with caregiver burdens and visual disability.
4. Demonstrate MSW-in-the-Loop decision flow:
   - **Accept**: Worker incorporates proposed question directly into the conversation.
   - **Edit**: Worker customizes proposed phrasing for patient rapport.
   - **Reject**: Worker discards suggested follow-up in favor of their clinical intuition.
5. Export conversational session traces to `data/interim/llm_interview/conversations/simulated_interview_traces.json`.


In [ ]:
from pathlib import Path
import json
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Any

def _find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / 'data').exists() and (p / 'experiments').exists():
            return p
    return Path('../../..').resolve()

PROJECT_ROOT = _find_project_root(Path.cwd())
CONV_DIR = PROJECT_ROOT / 'data' / 'interim' / 'llm_interview' / 'conversations'
CONV_DIR.mkdir(parents=True, exist_ok=True)

TRACE_OUT_PATH = CONV_DIR / 'simulated_interview_traces.json'
print(f"Conversation directory: {CONV_DIR}")


### 2. Dialogue State Tracker Implementation
The conversation state tracker maintains a structured history of:
- Current active domain
- Verified facts confirmed by the MSW
- Verbatim patient utterance history
- Candidate follow-up questions generated by the LLM
- MSW triage action (`ACCEPTED`, `EDITED`, `REJECTED`)


In [ ]:
@dataclass
class DialogueTurn:
    turn_id: int
    speaker: str  # 'MSW', 'PATIENT', 'ASSISTANT_SUGGESTION'
    text: str
    target_entity: Optional[str] = None
    msw_action: Optional[str] = None  # 'ACCEPTED', 'EDITED', 'REJECTED', 'VERBATIM'
    edited_text: Optional[str] = None

@dataclass
class InterviewSessionState:
    session_id: str
    patient_anon_id: str
    current_domain: str
    confirmed_entities: Dict[str, Any] = field(default_factory=dict)
    conversation_history: List[DialogueTurn] = field(default_factory=list)
    pending_followups: List[Dict[str, str]] = field(default_factory=list)

    def add_patient_response(self, text: str):
        turn = DialogueTurn(
            turn_id=len(self.conversation_history) + 1,
            speaker='PATIENT',
            text=text,
            msw_action='VERBATIM'
        )
        self.conversation_history.append(turn)

    def record_msw_decision(self, suggestion_id: str, action: str, final_text: str, target_entity: str):
        assert action in {'ACCEPTED', 'EDITED', 'REJECTED'}, f"Invalid MSW action: {action}"
        turn = DialogueTurn(
            turn_id=len(self.conversation_history) + 1,
            speaker='MSW',
            text=final_text,
            target_entity=target_entity,
            msw_action=action,
            edited_text=final_text if action == 'EDITED' else None
        )
        self.conversation_history.append(turn)

print("Dialogue State Tracker classes initialized successfully.")


### 3. Contextual Follow-up Generation Rule Engine
To demonstrate the deterministic execution of the LLM copilot pipeline without external API calls, we implement the core reasoning rule engine matching the LLM prompt specification.


In [ ]:
def generate_contextual_followups(session: InterviewSessionState) -> List[Dict[str, str]]:
    """
    Simulates the LLM follow-up generation logic specified in Notebook 02.
    Evaluates missing entities against the confirmed state and latest patient response.
    """
    last_patient_turns = [t for t in session.conversation_history if t.speaker == 'PATIENT']
    if not last_patient_turns:
        return []
    latest_text = last_patient_turns[-1].text.lower()
    followups = []

    if session.current_domain == 'financial_circumstances':
        # Check if income mentioned without exact figures
        if any(w in latest_text for w in ['pasada', 'tricycle', 'arawan', 'benta']) and 'monthly_income' not in session.confirmed_entities:
            followups.append({
                "suggestion_id": "SUGG-FIN-01",
                "target_entity": "monthly_income_estimate",
                "suggested_taglish": "Mga magkano po karaniwan ang naiuuwi ninyo sa pamamasada kada araw kapag nabawas na po ang boundary at krudo?",
                "suggested_en": "Approximately how much do you take home daily from driving after subtracting boundary and fuel expenses?",
                "rationale": "Clarify informal daily net take-home earnings."
            })
        if any(w in latest_text for w in ['utang', 'tulong', 'hiram', 'ospital']) and 'outstanding_debt' not in session.confirmed_entities:
            followups.append({
                "suggestion_id": "SUGG-FIN-02",
                "target_entity": "outstanding_debt_amount",
                "suggested_taglish": "May natitira pa po bang balanse sa inyong mga naunang gamutan o utang na kailangang bayaran buwan-buwan?",
                "suggested_en": "Is there any remaining debt or hospital balance from prior treatments that requires monthly repayment?",
                "rationale": "Assess debt burden and existing hospital liabilities."
            })

    elif session.current_domain == 'household_and_family':
        if any(w in latest_text for w in ['anak', 'bata', 'matanda', 'nanay']) and 'number_of_dependents' not in session.confirmed_entities:
            followups.append({
                "suggestion_id": "SUGG-FAM-01",
                "target_entity": "dependent_breakdown",
                "suggested_taglish": "Ilan po sa inyong mga kasama sa bahay ang nag-aaral pa o senior citizen na kailangan ng espesyal na alaga?",
                "suggested_en": "How many household members are currently studying or senior citizens needing special care?",
                "rationale": "Detail the composition of dependents."
            })

    session.pending_followups = followups
    return followups

print("Follow-up generation logic defined.")


### 4. End-to-End Multi-Turn Interview Simulation
We simulate an authentic multi-turn intake session with an anonymized synthetic cataract patient (`SYN-PAT-0042`).
The caseworker navigates through Financial and Household domains with interactive triage.


In [ ]:
# Initialize session
session = InterviewSessionState(
    session_id="SESS-20261123-001",
    patient_anon_id="SYN-PAT-0042",
    current_domain="financial_circumstances"
)

# Turn 1: MSW asks initial exploratory financial question
session.record_msw_decision(
    suggestion_id="INITIAL-FIN",
    action="ACCEPTED",
    final_text="Kamusta po ang inyong hanapbuhay sa kasalukuyan at sino po ang sumusuporta sa inyong pamilya?",
    target_entity="livelihood_overview"
)

# Turn 2: Patient responds with informal livelihood
session.add_patient_response(
    "Namasada po ako ng traysikel dati ma'am, pero ngayon dahil malabo na ang kanang mata ko, 2 hanggang 3 araw na lang ako nakakabiyahe. Yung asawa ko naman nagtitinda-tinda ng meryenda sa labas ng bahay."
)

# Turn 3: Assistant generates contextual follow-ups
suggs = generate_contextual_followups(session)
print(f"Assistant generated {len(suggs)} follow-up suggestions:")
for s in suggs:
    print(f" - [{s['suggestion_id']}] {s['suggested_taglish']}")

# Turn 4: MSW edits suggestion 1 to match rapport
selected_sugg = suggs[0]
msw_edited_text = "Naiintindihan ko po tatay. Sa 2 hanggang 3 araw po na nakakabiyahe kayo, mga magkano po ang naiuuwi ninyo sa pamilya kada linggo?"
session.record_msw_decision(
    suggestion_id=selected_sugg['suggestion_id'],
    action="EDITED",
    final_text=msw_edited_text,
    target_entity=selected_sugg['target_entity']
)

# Turn 5: Patient clarifies income amount
session.add_patient_response(
    "Mga 1,200 hanggang 1,500 po kada linggo ang naiuuwi ko ma'am. Yung benta naman ng misis ko mga 200 pesos lang ang tubo sa isang araw, pambili lang ng bigas at ulam."
)
session.confirmed_entities['monthly_family_income_estimate'] = 8500.00
session.confirmed_entities['income_sources'] = ["irregular tricycle driving", "small snack vending"]

# Save trace artifact
serialized_history = [asdict(t) for t in session.conversation_history]
trace_payload = {
    "session_id": session.session_id,
    "patient_anon_id": session.patient_anon_id,
    "total_turns": len(session.conversation_history),
    "confirmed_entities": session.confirmed_entities,
    "turns": serialized_history
}

TRACE_OUT_PATH.write_text(json.dumps(trace_payload, indent=2), encoding='utf-8')
print(f"Exported simulated interview trace ({len(serialized_history)} turns) -> {TRACE_OUT_PATH}")


### Summary of Notebook 03 Deliverables
- [x] Implemented multi-turn dialogue state tracking (`InterviewSessionState`, `DialogueTurn`).
- [x] Demonstrated MSW-in-the-loop decision capture (`ACCEPTED`, `EDITED`, `REJECTED`).
- [x] Verified contextual follow-up generation for informal Taglish responses.
- [x] Exported serialized conversation trace artifact (`simulated_interview_traces.json`).
